**Import packages**

In [17]:
import pandas as pd 
import matplotlib.pyplot as plt
import numpy as np
import statsmodels.api as sm

**Import datas: PWI, DP, DG, EG, GDPG**

In [18]:
pwi_data = pd.read_csv("C:\\Users\\王亭烜\\Desktop\\Thesis\\Data\\data1221\\EstimatedStates1.csv")
lty_data = pd.read_csv("C:\\Users\\王亭烜\\Desktop\\Thesis\\Data\\data1221\\DGS10.csv")
gdp_data = pd.read_csv("C:\\Users\\王亭烜\\Desktop\\Thesis\\Data\\data1221\\GDP.csv")
earnings_data = pd.read_excel("C:\\Users\\王亭烜\\Desktop\\Thesis\\Data\\data1221\\earnings.xlsx") 
dp_data = pd.read_excel("C:\\Users\\王亭烜\\Desktop\\Thesis\\Data\\data1221\\dividend_price_ratio.xlsx")
ltr_data = pd.read_excel("C:\\Users\\王亭烜\\Desktop\\Thesis\\Data\\data1221\\ltr_data.xlsx")
ltr_data 

,yyyymm,ltr
0,192612,0.007800
1,192701,0.007500
2,192702,0.008800
3,192703,0.025300
4,192704,-0.000500
...,...,...
1159,202307,-0.003540
1160,202308,-0.005182
1161,202309,-0.022094
1162,202310,-0.012083


In [19]:
### dividend-price ratio and dividend growth must be calculated
### https://quant.stackexchange.com/questions/48643/sp-500-dividend-data
### the data is fetched from CRSP database 
sp_data = pd.read_csv("C:\\Users\\王亭烜\\Desktop\\Thesis\\Data\\data1221\\sp_monthly.csv")
sp_data['dp'] = (1+sp_data['vwretd']) / (1+sp_data['vwretx']) - 1  
sp_data['dp_lag'] = sp_data['dp'].shift(1)
sp_data['spindx_lag1'] = sp_data['spindx'].shift(1)
sp_data['dg'] = sp_data['dp'] / sp_data['dp_lag'] * sp_data['spindx'] / sp_data['spindx_lag1']

dg_data = pd.DataFrame(columns=['Date', 'DG', 'DP'])
dg_data['Date'] = sp_data['caldt']
dg_data['DG'] = sp_data['dg']
dg_data['DP'] = sp_data['dp']

**Cash flow and discount rate predictability**

In [20]:
### change the format of the date column to merge all the data
# probability weighting index data 
pwi_data['Date'] = pd.to_datetime(pwi_data['Date'])
pwi_data['year_month'] = pwi_data['Date'].dt.strftime('%Y-%m')

# dividend price ratio
dp_data['year_month'] = dp_data['yyyymm'].astype(str).str[:4] + '-' + dp_data['yyyymm'].astype(str).str[4:]

# dividend growth data 
dg_data['Date'] = pd.to_datetime(dg_data['Date'])
dg_data['year_month'] = dg_data['Date'].dt.strftime('%Y-%m')
dg_data 

# long-term government bond return data 
ltr_data['year_month'] = ltr_data['yyyymm'].astype(str).str[:4] + '-' + ltr_data['yyyymm'].astype(str).str[4:]

# 10-year treasury bond yield data is fetched from FRED website 
# https://fred.stlouisfed.org/series/DGS10
lty_data['observation_date'] = pd.to_datetime(lty_data['observation_date'])
lty_data.set_index('observation_date', inplace=True)
lty_data = lty_data.resample('M').last().reset_index() 
lty_data['year_month'] = lty_data['observation_date'].dt.strftime('%Y-%m')

# GDP data is fetched from FRED website 
# https://fred.stlouisfed.org/series/GDP
gdp_data['observation_date'] = pd.to_datetime(gdp_data['observation_date'])
gdp_data['observation_date'] = gdp_data['observation_date'] - pd.Timedelta(days=1)
gdp_data['year_month'] = gdp_data['observation_date'].dt.strftime('%Y-%m')
gdp_data['log_GDP'] = np.log(gdp_data['GDP'])
gdp_data['GDPG'] = gdp_data['log_GDP'].diff() 
gdp_data

# SPX 500 earnings 
# the data is fetched from Bloomberg (quarterly basis)
earnings_data['year_month'] = earnings_data['year_month'].dt.strftime('%Y-%m')
earnings_data['log_E'] = np.log(earnings_data['earnings'])
earnings_data['EG'] = earnings_data['log_E'].diff() 

dp_data 


C:\Users\王亭烜\AppData\Local\Temp\ipykernel_29552\2310134033.py:21: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  lty_data = lty_data.resample('M').last().reset_index()


,yyyymm,log_dp,year_month
0,192612,-2.973012,1926-12
1,192701,-2.942374,1927-01
2,192702,-2.979535,1927-02
3,192703,-2.976535,1927-03
4,192704,-2.984225,1927-04
...,...,...,...
1159,202307,-4.198544,2023-07
1160,202308,-4.177780,2023-08
1161,202309,-4.124953,2023-09
1162,202310,-4.097976,2023-10


In [21]:
### Does probability weighting index predict future expected cash flow or discount rate?
merged_data = pd.merge(pwi_data, dp_data, on='year_month', how='inner')
merged_data = pd.merge(merged_data, dg_data, on='year_month', how='inner')
merged_data['DP_lead'] = merged_data['log_dp'].shift(-1)
reg_data = merged_data.dropna(subset=['DP_lead', 'Alpha', 'log_dp'])

X = reg_data[['Alpha', 'log_dp']]
y = reg_data['DP_lead']
X = sm.add_constant(X)
maxlags = 1 
model = sm.OLS(y, X).fit(cov_type='HAC', cov_kwds={'maxlags': maxlags})
model.summary() 

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                DP_lead   R-squared:                       0.028
Model:                            OLS   Adj. R-squared:                  0.015
Method:                 Least Squares   F-statistic:                     2.455
Date:                Sun, 09 Mar 2025   Prob (F-statistic):             0.0892
Time:                        14:11:43   Log-Likelihood:                 99.295
No. Observations:                 158   AIC:                            -192.6
Df Residuals:                     155   BIC:                            -183.4
Df Model:                           2                                         
Covariance Type:                  HAC                                         
==============================================================================
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const         -3.9347      0.052    -76.382      0.000      -4.036      -3.834
Alpha         -0.0670      0.049     -1.373      0.170      -0.163       0.029
DG             0.0172      0.010      1.692      0.091      -0.003       0.037
==============================================================================
Omnibus:                       30.671   Durbin-Watson:                   0.118
Prob(Omnibus):                  0.000   Jarque-Bera (JB):               41.350
Skew:                          -1.160   Prob(JB):                     1.05e-09
Kurtosis:                       3.949   Cond. No.                         10.3
==============================================================================

Notes:
[1] Standard Errors are heteroscedasticity and autocorrelation robust (HAC) using 1 lags and without small sample correction
"""

In [14]:
### Does probability weighting index predict future expected cash flow or discount rate?
merged_data = pd.merge(pwi_data, dg_data, on='year_month', how='inner')
merged_data = pd.merge(merged_data, dp_data, on='year_month', how='inner')
merged_data['DP_lead'] = merged_data['log_dp'].shift(-1)
merged_data['DG_lead'] = merged_data['DG'].shift(-1)
reg_data = merged_data.dropna(subset=['DG_lead', 'Alpha', 'log_dp', 'DG'])

X = reg_data[['Alpha', 'DG']]
y = reg_data['DG_lead']
X = sm.add_constant(X)
maxlags = 1 
model = sm.OLS(y, X).fit(cov_type='HAC', cov_kwds={'maxlags': maxlags})
model.summary() 

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                DG_lead   R-squared:                       0.238
Model:                            OLS   Adj. R-squared:                  0.228
Method:                 Least Squares   F-statistic:                     94.80
Date:                Sun, 09 Mar 2025   Prob (F-statistic):           1.29e-27
Time:                        14:09:53   Log-Likelihood:                -127.41
No. Observations:                 158   AIC:                             260.8
Df Residuals:                     155   BIC:                             270.0
Df Model:                           2                                         
Covariance Type:                  HAC                                         
==============================================================================
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const          1.7967      0.160     11.206      0.000       1.482       2.111
Alpha         -0.0825      0.133     -0.622      0.534      -0.342       0.178
DG            -0.4890      0.036    -13.729      0.000      -0.559      -0.419
==============================================================================
Omnibus:                       50.919   Durbin-Watson:                   2.908
Prob(Omnibus):                  0.000   Jarque-Bera (JB):               10.789
Skew:                           0.291   Prob(JB):                      0.00454
Kurtosis:                       1.860   Cond. No.                         10.3
==============================================================================

Notes:
[1] Standard Errors are heteroscedasticity and autocorrelation robust (HAC) using 1 lags and without small sample correction
"""

In [15]:
### Alternative proxy for discount rate and expected cash flow 
### Follow Wang et al. (2019), we use long term bond yield as proxy for discount rate and earnings growth as proxy for expected cash flow 

# Does probability weighting index predict LTY?
merged_data = pd.merge(pwi_data, dg_data, on='year_month', how='inner')
merged_data = pd.merge(merged_data, lty_data, on='year_month', how='inner')
merged_data['DGS10_lead'] = merged_data['DGS10'].shift(-1)
reg_data = merged_data.dropna(subset=['DGS10_lead', 'Alpha', 'DG', 'DGS10'])

X = reg_data[['Alpha', 'DG']]
y = reg_data['DGS10_lead']
X = sm.add_constant(X)
maxlags = 1 
model = sm.OLS(y, X).fit(cov_type='HAC', cov_kwds={'maxlags': maxlags})
model.summary() 


<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:             DGS10_lead   R-squared:                       0.080
Model:                            OLS   Adj. R-squared:                  0.069
Method:                 Least Squares   F-statistic:                     4.026
Date:                Sun, 09 Mar 2025   Prob (F-statistic):             0.0197
Time:                        14:10:19   Log-Likelihood:                -174.76
No. Observations:                 158   AIC:                             355.5
Df Residuals:                     155   BIC:                             364.7
Df Model:                           2                                         
Covariance Type:                  HAC                                         
==============================================================================
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const          3.0562      0.293     10.436      0.000       2.482       3.630
Alpha         -0.7731      0.274     -2.820      0.005      -1.310      -0.236
DG             0.0055      0.065      0.085      0.932      -0.121       0.132
==============================================================================
Omnibus:                        5.314   Durbin-Watson:                   0.133
Prob(Omnibus):                  0.070   Jarque-Bera (JB):                5.443
Skew:                           0.437   Prob(JB):                       0.0658
Kurtosis:                       2.750   Cond. No.                         10.3
==============================================================================

Notes:
[1] Standard Errors are heteroscedasticity and autocorrelation robust (HAC) using 1 lags and without small sample correction
"""

In [16]:
# Does probability weighting index predict log earnings growth?
merged_data = pd.merge(pwi_data, dg_data, on='year_month', how='inner')
merged_data = pd.merge(merged_data, earnings_data, on='year_month', how='inner')
merged_data['EG_lead'] = merged_data['EG'].shift(-1)
reg_data = merged_data.dropna(subset=['EG_lead', 'Alpha', 'DG'])

X = reg_data[['Alpha', 'DG']]
y = reg_data['EG_lead']
X = sm.add_constant(X)
maxlags = 1 
model = sm.OLS(y, X).fit(cov_type='HAC', cov_kwds={'maxlags': maxlags})
model.summary() 

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                EG_lead   R-squared:                       0.106
Model:                            OLS   Adj. R-squared:                  0.070
Method:                 Least Squares   F-statistic:                     1.307
Date:                Sun, 09 Mar 2025   Prob (F-statistic):              0.280
Time:                        14:10:39   Log-Likelihood:                 62.297
No. Observations:                  52   AIC:                            -118.6
Df Residuals:                      49   BIC:                            -112.7
Df Model:                           2                                         
Covariance Type:                  HAC                                         
==============================================================================
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const          0.0542      0.071      0.767      0.443      -0.084       0.193
Alpha         -0.0940      0.060     -1.575      0.115      -0.211       0.023
DG             0.0912      0.103      0.889      0.374      -0.110       0.292
==============================================================================
Omnibus:                       42.559   Durbin-Watson:                   1.969
Prob(Omnibus):                  0.000   Jarque-Bera (JB):              217.949
Skew:                           1.981   Prob(JB):                     4.71e-48
Kurtosis:                      12.214   Cond. No.                         19.8
==============================================================================

Notes:
[1] Standard Errors are heteroscedasticity and autocorrelation robust (HAC) using 1 lags and without small sample correction
"""

In [270]:
### We only have GDP data in quarter frequency (does not match with our results)  
merged_data = pd.merge(pwi_data, dp_data, on='year_month', how='inner')
merged_data = pd.merge(merged_data, gdp_data, on='year_month', how='inner')
merged_data['GDPG_lead'] = merged_data['GDPG'].shift(-1)
reg_data = merged_data.dropna(subset=['GDPG_lead', 'Alpha', 'log_dp'])

X = reg_data[['Alpha', 'log_dp']]
y = reg_data['GDPG_lead']
X = sm.add_constant(X)
maxlags = 1 
model = sm.OLS(y, X).fit(cov_type='HAC', cov_kwds={'maxlags': maxlags})
model.summary() 

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:              GDPG_lead   R-squared:                       0.156
Model:                            OLS   Adj. R-squared:                  0.122
Method:                 Least Squares   F-statistic:                     5.343
Date:                Mon, 03 Mar 2025   Prob (F-statistic):            0.00796
Time:                        12:34:07   Log-Likelihood:                 138.35
No. Observations:                  52   AIC:                            -270.7
Df Residuals:                      49   BIC:                            -264.8
Df Model:                           2                                         
Covariance Type:                  HAC                                         
==============================================================================
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const         -0.0602      0.060     -1.005      0.315      -0.178       0.057
Alpha          0.0255      0.013      1.935      0.053      -0.000       0.051
log_dp        -0.0114      0.017     -0.654      0.513      -0.045       0.023
==============================================================================
Omnibus:                       74.294   Durbin-Watson:                   2.133
Prob(Omnibus):                  0.000   Jarque-Bera (JB):             1093.824
Skew:                          -3.635   Prob(JB):                    3.01e-238
Kurtosis:                      24.260   Cond. No.                         132.
==============================================================================

Notes:
[1] Standard Errors are heteroscedasticity and autocorrelation robust (HAC) using 1 lags and without small sample correction
"""